# Final corrected PEFT v4 test evaluation — frozen Condition C

This notebook performs the single final test evaluation of the validation-selected corrected PEFT configuration:

- completed answer-only v4 adapter;
- exact prompt construction used during v4 training;
- `add_special_tokens=False`, leaving exactly one BOS token from the chat template;
- deterministic generation and the unchanged strict parser.

Condition C was selected before this run using validation macro F1 (`0.8550`; validation accuracy `0.9256`). This notebook evaluates no alternative prompt, performs no training, and must not be used for further model selection. Predictions are checkpointed to Google Drive and resume safely after interruption.


## 1. Environment, selection record and frozen test split


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

%pip install -q transformers accelerate bitsandbytes peft scikit-learn


In [ ]:
import gc
import json
import os
import random
import re
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import transformers
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DATA_PATH = "/content/drive/MyDrive/Thesis1_Project"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
ADAPTER_DIR = os.path.join(DATA_PATH, "peft_answer_only_control_v4_final_adapter")
SEED = 42

VALIDATION_DECISION = os.path.join(DATA_PATH, "peft_v4_validation_diagnostic_metadata.json")
VALIDATION_SUMMARY = os.path.join(DATA_PATH, "peft_v4_validation_diagnostic_comparison.csv")
TEST_PARTIAL = os.path.join(DATA_PATH, "peft_v4_condition_c_test_partial.csv")
TEST_FINAL = os.path.join(DATA_PATH, "peft_v4_condition_c_test_final.csv")
TEST_METRICS = os.path.join(DATA_PATH, "peft_v4_condition_c_test_metrics.json")
TEST_PER_CLASS = os.path.join(DATA_PATH, "peft_v4_condition_c_test_per_class.csv")
TEST_CONFUSION = os.path.join(DATA_PATH, "peft_v4_condition_c_test_confusion_counts.csv")
TEST_CONFLICT = os.path.join(DATA_PATH, "peft_v4_condition_c_test_conflict_analysis.csv")
TEST_METADATA = os.path.join(DATA_PATH, "peft_v4_condition_c_test_metadata.json")

SELECTED_CONDITION = "C_training_prompt_single_bos"
TRAINING_INSTRUCTION = "Classify the following contract clause into its correct clause type:"

CHOICES = [
    "Parties", "License Grant", "Cap On Liability", "Audit Rights",
    "Anti-Assignment", "Insurance", "Document Name", "Agreement Date",
    "Governing Law", "Expiration Date", "Effective Date",
    "Post-Termination Services", "Revenue/Profit Sharing",
    "Minimum Commitment", "Exclusivity", "Rofr/Rofo/Rofn",
    "Ip Ownership Assignment", "Non-Transferable License", "Non-Compete",
    "Termination For Convenience",
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")
torch.cuda.manual_seed_all(SEED)

required_files = [
    os.path.join(DATA_PATH, "test.csv"),
    os.path.join(ADAPTER_DIR, "adapter_config.json"),
    VALIDATION_DECISION,
    VALIDATION_SUMMARY,
]
missing = [path for path in required_files if not os.path.exists(path)]
if missing:
    raise FileNotFoundError("Required frozen files are missing:\n" + "\n".join(missing))

with open(VALIDATION_DECISION, "r", encoding="utf-8") as file:
    validation_decision = json.load(file)
if validation_decision.get("best_validation_condition_by_macro_f1") != SELECTED_CONDITION:
    raise RuntimeError("The saved validation decision does not select frozen Condition C.")
if validation_decision.get("test_evaluated") is not False:
    raise RuntimeError("The validation decision record does not confirm that test was untouched.")

validation_summary = pd.read_csv(VALIDATION_SUMMARY)
selected_validation = validation_summary.loc[
    validation_summary["condition"].eq(SELECTED_CONDITION)
]
if len(selected_validation) != 1:
    raise RuntimeError("Condition C is missing or duplicated in the validation summary.")
selected_validation = selected_validation.iloc[0]
if not np.isclose(float(selected_validation["macro_f1"]), 0.854972291860513):
    raise RuntimeError("Condition C validation macro F1 does not match the frozen decision record.")

test_df = pd.read_csv(os.path.join(DATA_PATH, "test.csv")).reset_index(drop=True)
required_columns = {"clause_text", "clause_type", "contract_title"}
missing_columns = required_columns - set(test_df.columns)
if missing_columns:
    raise ValueError(f"test.csv is missing columns: {sorted(missing_columns)}")
if len(test_df) != 1850:
    raise ValueError(f"Expected 1,850 frozen test rows, found {len(test_df):,}.")
if set(test_df["clause_type"].unique()) != set(CHOICES):
    raise ValueError("Test labels do not match the frozen 20-class schema.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Transformers: {transformers.__version__}")
print(f"Frozen selected condition: {SELECTED_CONDITION}")
print(f"Validation macro F1 used for selection: {float(selected_validation['macro_f1']):.4f}")
print(f"Frozen test rows: {len(test_df):,}")
print("Selection record verified. No alternative test condition will be evaluated.")


## 2. Exact training-matched prompt and BOS audit


In [ ]:
def build_chat_prompt(instruction):
    messages = [{"role": "user", "content": instruction}]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    return f"<s>[INST] {instruction} [/INST]"


def build_training_matched_prompt(clause_text):
    instruction = (
        f"{TRAINING_INSTRUCTION}\n\n"
        f"Clause:\n{clause_text}\n\n"
        "Return ONLY the category name. Do not explain your answer and do not add punctuation."
    )
    return build_chat_prompt(instruction)


sample_prompt = build_training_matched_prompt(test_df.loc[0, "clause_text"])
sample_ids = tokenizer(sample_prompt, add_special_tokens=False)["input_ids"]
sample_bos_count = sample_ids.count(tokenizer.bos_token_id)

print("Prompt preview:\n")
print(sample_prompt[:900])
print("\nFirst tokens:", tokenizer.convert_ids_to_tokens(sample_ids[:10]))
print("BOS count:", sample_bos_count)

if sample_bos_count != 1:
    raise RuntimeError("The frozen Condition C prompt must contain exactly one BOS token.")
print("Prompt audit passed: exact training wording and one BOS token.")


## 3. Single final test generation

The partial file is updated after every batch. If Colab disconnects, rerun the notebook from the beginning; completed row IDs will be skipped. Do not rename an incomplete partial file to the final filename.


In [ ]:
def parse_prediction(raw_output):
    if raw_output is None or pd.isna(raw_output):
        return "Unknown"
    text = str(raw_output).strip()
    if not text:
        return "Unknown"
    candidates = [text, text.splitlines()[0].strip()]
    for candidate in candidates:
        candidate = re.sub(r"^[\s`*_\"'“”‘’]+|[\s`*_\"'“”‘’]+$", "", candidate).strip()
        category_match = re.fullmatch(r"(?i)category\s*:\s*(.+)", candidate)
        if category_match:
            candidate = category_match.group(1).strip()
        candidate = re.sub(r"[.,;:!?]+$", "", candidate).strip()
        for choice in CHOICES:
            if candidate.casefold() == choice.casefold():
                return choice
    return "Unknown"


def validate_test_predictions(frame):
    frame = frame.copy()
    frame["row_id"] = pd.to_numeric(frame["row_id"], errors="raise").astype(int)
    frame = frame.drop_duplicates("row_id", keep="last").sort_values("row_id")
    if len(frame) != len(test_df) or set(frame["row_id"]) != set(range(len(test_df))):
        raise RuntimeError("Final predictions are incomplete or row IDs are misaligned.")
    if not frame["clause_text"].astype(str).reset_index(drop=True).equals(
        test_df["clause_text"].astype(str)
    ):
        raise RuntimeError("Final prediction text is misaligned with test.csv.")
    if not frame["clause_type"].astype(str).reset_index(drop=True).equals(
        test_df["clause_type"].astype(str)
    ):
        raise RuntimeError("Final reference labels are misaligned with test.csv.")
    return frame


if os.path.exists(TEST_FINAL):
    test_results = validate_test_predictions(pd.read_csv(TEST_FINAL))
    print("Complete final Condition C test predictions already exist; generation will be skipped.")
elif os.path.exists(TEST_PARTIAL):
    test_results = pd.read_csv(TEST_PARTIAL)
    test_results["row_id"] = pd.to_numeric(
        test_results["row_id"], errors="coerce"
    ).astype("Int64")
    test_results = (
        test_results.dropna(subset=["row_id"])
        .drop_duplicates("row_id", keep="last")
        .sort_values("row_id")
    )
    print(f"Resuming final test generation at {len(test_results):,}/{len(test_df):,} rows.")
else:
    test_results = pd.DataFrame(
        columns=["row_id", "clause_text", "clause_type", "predicted_label", "raw_output"]
    )
    print("Starting the frozen Condition C test evaluation from row zero.")

completed_ids = set(test_results["row_id"].astype(int)) if len(test_results) else set()
pending = test_df.loc[~test_df.index.isin(completed_ids)].copy()
print(f"Completed: {len(test_results):,}; remaining: {len(pending):,}")


In [ ]:
if len(pending):
    for variable_name in ["model", "base_model", "evaluation_model"]:
        if variable_name in globals():
            del globals()[variable_name]
    gc.collect()
    torch.cuda.empty_cache()

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
        dtype=torch.float16,
    )
    evaluation_model = PeftModel.from_pretrained(
        base_model,
        ADAPTER_DIR,
        is_trainable=False,
    )
    evaluation_model.eval()
    evaluation_model.config.use_cache = True

    batch_size = 4
    for start in range(0, len(pending), batch_size):
        batch = pending.iloc[start:start + batch_size]
        prompts = [
            build_training_matched_prompt(text)
            for text in batch["clause_text"]
        ]
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=4096,
            add_special_tokens=False,
        ).to("cuda")

        for row_ids, attention in zip(inputs["input_ids"], inputs["attention_mask"]):
            active_ids = row_ids[attention.bool()].tolist()
            if active_ids.count(tokenizer.bos_token_id) != 1:
                raise RuntimeError("A test prompt does not contain exactly one BOS token.")

        with torch.inference_mode():
            generated = evaluation_model.generate(
                **inputs,
                max_new_tokens=12,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

        generated_tokens = generated[:, inputs["input_ids"].shape[1]:]
        raw_outputs = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        rows = []
        for row_id, (_, row), raw_output in zip(batch.index, batch.iterrows(), raw_outputs):
            rows.append({
                "row_id": int(row_id),
                "clause_text": row["clause_text"],
                "clause_type": row["clause_type"],
                "predicted_label": parse_prediction(raw_output),
                "raw_output": raw_output.strip(),
            })

        test_results = pd.concat([test_results, pd.DataFrame(rows)], ignore_index=True)
        test_results["row_id"] = test_results["row_id"].astype(int)
        test_results = (
            test_results.drop_duplicates("row_id", keep="last")
            .sort_values("row_id")
        )
        test_results.to_csv(TEST_PARTIAL, index=False)
        if len(test_results) % 100 < batch_size or len(test_results) == len(test_df):
            print(f"Final Condition C test: {len(test_results):,}/{len(test_df):,}")

    del evaluation_model, base_model
    gc.collect()
    torch.cuda.empty_cache()

test_results = validate_test_predictions(test_results)
test_results.to_csv(TEST_FINAL, index=False)
print("Final predictions saved:", TEST_FINAL)


## 4. Frozen test metrics and class-level results


In [ ]:
y_true = test_results["clause_type"]
y_pred = test_results["predicted_label"]

metrics = {
    "method": "Corrected PEFT v4 — Condition C",
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_precision": float(
        precision_score(y_true, y_pred, labels=CHOICES, average="macro", zero_division=0)
    ),
    "macro_recall": float(
        recall_score(y_true, y_pred, labels=CHOICES, average="macro", zero_division=0)
    ),
    "macro_f1": float(
        f1_score(y_true, y_pred, labels=CHOICES, average="macro", zero_division=0)
    ),
    "weighted_f1": float(
        f1_score(y_true, y_pred, labels=CHOICES, average="weighted", zero_division=0)
    ),
    "unknown_count": int(y_pred.eq("Unknown").sum()),
    "unknown_rate": float(y_pred.eq("Unknown").mean()),
    "rows": int(len(test_results)),
    "validation_selected_condition": SELECTED_CONDITION,
    "test_evaluated_once": True,
}
with open(TEST_METRICS, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

display(pd.DataFrame([metrics]).round(4))

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=CHOICES, zero_division=0
)
per_class = pd.DataFrame({
    "clause_type": CHOICES,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support.astype(int),
})
per_class.to_csv(TEST_PER_CLASS, index=False)
display(per_class.round(4))

confusion = pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=CHOICES + ["Unknown"]),
    index=pd.Index(CHOICES + ["Unknown"], name="true_label"),
    columns=pd.Index(CHOICES + ["Unknown"], name="predicted_label"),
)
confusion.to_csv(TEST_CONFUSION)

print("\nClassification report:\n")
print(classification_report(y_true, y_pred, labels=CHOICES, zero_division=0))
print("Saved metrics:", TEST_METRICS)
print("Saved per-class results:", TEST_PER_CLASS)
print("Saved confusion counts:", TEST_CONFUSION)


## 5. Exact-text conflict sensitivity on the frozen test predictions


In [ ]:
label_counts_by_text = test_df.groupby("clause_text")["clause_type"].nunique()
conflicting_texts = set(label_counts_by_text[label_counts_by_text > 1].index)
conflict_flag = test_df["clause_text"].isin(conflicting_texts).to_numpy()

conflict_rows = []
for subset_name, mask in [
    ("all", np.ones(len(test_df), dtype=bool)),
    ("exact_text_conflict", conflict_flag),
    ("no_exact_text_conflict", ~conflict_flag),
]:
    subset = test_results.loc[mask]
    present_labels = sorted(subset["clause_type"].unique())
    conflict_rows.append({
        "subset": subset_name,
        "rows": int(len(subset)),
        "labels_present": int(len(present_labels)),
        "accuracy": float(accuracy_score(subset["clause_type"], subset["predicted_label"])),
        "macro_f1_present_labels": float(f1_score(
            subset["clause_type"],
            subset["predicted_label"],
            labels=present_labels,
            average="macro",
            zero_division=0,
        )),
        "weighted_f1": float(f1_score(
            subset["clause_type"],
            subset["predicted_label"],
            labels=present_labels,
            average="weighted",
            zero_division=0,
        )),
    })

conflict_analysis = pd.DataFrame(conflict_rows)
conflict_analysis.to_csv(TEST_CONFLICT, index=False)
display(conflict_analysis.round(4))

ceiling = (
    test_df.groupby(["clause_text", "clause_type"])
    .size()
    .groupby(level=0)
    .max()
    .sum()
    / len(test_df)
)
print(f"Conflicting test texts: {len(conflicting_texts)}")
print(f"Rows carrying a conflicted exact text: {int(conflict_flag.sum())}/{len(test_df)}")
print(f"Exact-text deterministic test accuracy ceiling: {ceiling:.4f}")
print("Saved conflict analysis:", TEST_CONFLICT)


## 6. Final execution record


In [ ]:
metadata = {
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "status": "final_corrected_peft_test_complete",
    "model_id": MODEL_ID,
    "adapter": ADAPTER_DIR,
    "selected_condition": SELECTED_CONDITION,
    "selection_metric": "validation macro F1",
    "selected_validation_accuracy": float(selected_validation["accuracy"]),
    "selected_validation_macro_f1": float(selected_validation["macro_f1"]),
    "prompt": "exact v4 training prompt",
    "add_special_tokens": False,
    "expected_bos_count": 1,
    "do_sample": False,
    "max_new_tokens": 12,
    "parser": "strict full-label parser",
    "test_rows": int(len(test_results)),
    "test_metrics": metrics,
    "alternative_test_conditions_evaluated": False,
    "training_performed": False,
    "further_model_selection_permitted": False,
}
with open(TEST_METADATA, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print(json.dumps(metadata, indent=2))
print("\nFinal corrected PEFT test evaluation complete.")
